# Top Brass: from a literal model to indexed data

This companion to Lecture 2 develops the same linear program in stages: a literal formulation, variable bounds, indexed trophies and dictionaries, and a fully indexed resource model using a NamedArray. Each baseline formulation solves the same problem; only the final, explicitly marked experiment changes the data.

**Prerequisites:** Julia with JuMP, HiGHS, and NamedArrays, and an IJulia kernel using that environment. Install packages in your Julia environment before running this notebook. Run the cells from top to bottom. No external data files are required.

**Source:** Top Brass Trophy Company, Example 5.1 in Rardin, *Optimization in Operations Research* (1998). 

## The problem and its mathematical model

Top Brass makes football and soccer trophies. Each football trophy earns 12 dollars in profit and each soccer trophy earns 9 dollars. Available resources and per-trophy requirements are:

| Resource | Football | Soccer | Available |
| --- | ---: | ---: | ---: |
| Wood (board feet) | 4 | 2 | 4,800 |
| Plaques | 1 | 1 | 1,750 |
| Brass footballs | 1 | 0 | 1,000 |
| Brass soccer balls | 0 | 1 | 1,500 |

Assuming every trophy produced can be sold, how many of each should be made? Let $f$ and $s$ denote football and soccer trophy x. The model is

$$
\begin{aligned}
\max_{f,s}\quad &12f+9s\\
\text{subject to}\quad &4f+2s\le4800 &&\text{(wood)}\\
&f+s\le1750 &&\text{(plaques)}\\
&f\le1000 &&\text{(brass footballs)}\\
&s\le1500 &&\text{(brass soccer balls)}\\
&f,s\ge0.
\end{aligned}
$$

This LP allows continuous production quantities. Its solution happens to be integral; rounding is not a general substitute for modeling integrality.

## 1. A first, literal JuMP model

Julia is the programming language, JuMP expresses the optimization model, and HiGHS solves it. `Model(HiGHS.Optimizer)` connects a new model to the solver. `set_silent` suppresses solver logs so we can focus on our report.

`@variable`, `@constraint`, and `@objective` mirror the algebra. The `@` indicates a Julia macro: here it lets JuMP interpret mathematical-looking expressions as model components. 

In [ ]:
using JuMP, HiGHS

literal_model = Model(HiGHS.Optimizer)
set_silent(literal_model)

@variable(literal_model, f >= 0)           # football trophies
@variable(literal_model, s >= 0)           # soccer trophies
@constraint(literal_model, f <= 1000)      # Upper bound on football trophy production
@constraint(literal_model, s <= 1500)       # Upper bound on soccer trophy production
@constraint(literal_model, 4f + 2s <= 4800)        # total board feet of wood
@constraint(literal_model, f + s <= 1750)          # total number of plaques
@objective(literal_model, Max, 12f + 9s)           # maximize profit

print(literal_model)

### Solve, check, then report

`optimize!` asks the solver to solve the model. 

The `!` at the end of optimize does not mean that you are excited to solve the model, although you might be.  The `!` is a Julia funciton name convention meaning that the function may modify its arguments.  In this case we are modifying the `Model` object `literal_model` by solving it.

We query `termination_status` separately, require a proven optimum for this LP, and check that primal values are available before reading them. `value` retrieves a variable's solution and `objective_value` retrieves the objective. The short-circuit expression `condition || error(...)` stops execution when the condition is false.

In [ ]:
optimize!(literal_model)
status = termination_status(literal_model)
println("Termination status: ", status)
status == OPTIMAL ||
    error("Expected a proven optimum; solver stopped with status $(status).")
is_solved_and_feasible(literal_model) ||
    error("No feasible optimal solution is available.")
println("Objective value: ", objective_value(literal_model))

println("Football trophies: ", value(f))
println("Soccer trophies: ", value(s))

The optimum is 650 football trophies and 1,100 soccer trophies, with profit of **17,700 dollars**. This uses all 4,800 board feet of wood and all 1,750 plaques. These are the same quantities found geometrically.

### A small helper for subsequent models

The solve/check/report sequence repeats, so we put it in a function. `solve_and_report(model)` takes a model as its argument and returns its objective value after checking the result. Keeping the helper here makes the notebook self-contained; it can be reused in other notebooks and later moved to a shared Julia file.

In [ ]:
function solve_and_report(model)
    optimize!(model)
    status = termination_status(model)
    println("Termination status: ", status)
    status == OPTIMAL ||
        error("Expected a proven optimum; solver stopped with status $(status).")
    is_solved_and_feasible(model) ||
        error("No feasible optimal solution is available.")
    objective = objective_value(model)
    println("Objective value: ", objective)
    return objective
end

## 2. Inventory limits as variable bounds

The football and soccer inventory constraints each involve just one variable. We can express them as upper bounds directly in `@variable`. The feasible region and objective are unchanged. Build a fresh model for this version so the earlier constraints are not retained accidentally.

We can use (over-write) the variables.  (As long as we don't care to later refer to the variables from the earlier models in this notebook).

Note also that we can name or label our constraints if we wish by giving a second argument to the `@constraint` macro

In [ ]:
bounded_model = Model(HiGHS.Optimizer)
set_silent(bounded_model)
@variable(bounded_model, 0 <= f <= 1000)
@variable(bounded_model, 0 <= s <= 1500)
@constraint(bounded_model, wood_limit, 4 * f + 2 * s <= 4800)
@constraint(bounded_model, plaque_limit, f + s <= 1750)
@objective(bounded_model, Max, 12 * f + 9 * s)

bounded_profit = solve_and_report(bounded_model)
println("Football trophies: ", value(f))
println("Soccer trophies: ", value(s))

## 3. Indexed trophies and data in dictionaries

Football and soccer are members of an index set of products. The Julia vector `products` stores their labels; `:football` and `:soccer` are Julia symbols used as keys. The `:` in front of the label changes the type in Julia to a `Symbol`, not a `String`.  We will use Symbols a lot in this course.
A `Dict` maps each key to a numerical value. These are ordinary Julia data structures, separate from JuMP's variables and constraints.

Here, we move the profit, requirements, and availability into a data cell. This intermediate model still names wood and plaques separately; the next version will index the resources too.

In [ ]:
products = [:football, :soccer]
profit = Dict(:football => 12, :soccer => 9)
wood_requirement = Dict(:football => 4, :soccer => 2)
plaque_requirement = Dict(:football => 1, :soccer => 1)
product_limit = Dict(:football => 1000, :soccer => 1500)
wood_available = 4800
plaques_available = 1750

### Indexing your objects

`@variable(..., x[p in products], ...)` creates an indexed JuMP container of variables.  You can access a component directly like `x[:football]`. 

The generator `sum(profit[p] * x[p] for p in products)` corresponds to $\sum_{p\in P}c_p x_p$.

Named `@expression` objects describe resource use and profit. They are algebraic expressions, not extra decision variables or extra constraints. We define each resource constraint once and can also evaluate these expressions after solving.
You do not need to create expressions for the left-hand-side of your constraints.  But sometimes it is convenient.

Here, we do it, so we can check how much wood we used and how many plaques we used in our optimal production schedule.

In [ ]:
indexed_model = Model(HiGHS.Optimizer)
set_silent(indexed_model)
@variable(indexed_model, 0 <= x[p in products] <= product_limit[p])
@expression(indexed_model, total_wood,
    sum(wood_requirement[p] * x[p] for p in products))
@expression(indexed_model, total_plaques,
    sum(plaque_requirement[p] * x[p] for p in products))
@expression(indexed_model, total_profit,
    sum(profit[p] * x[p] for p in products))
@constraint(indexed_model, wood_limit, total_wood <= wood_available)
@constraint(indexed_model, plaque_limit, total_plaques <= plaques_available)
@objective(indexed_model, Max, total_profit)

indexed_profit = solve_and_report(indexed_model)
for p in products
    println(p, " trophies: ", value(x[p]))
end
println("Wood used: ", value(total_wood), " board feet")
println("Plaques used: ", value(total_plaques))

## 4. Resources in rows, products in columns

Now collect **all four resources** into a labeled recipe matrix. Entry $a_{rp}$ is the amount of resource $r$ needed for one unit of product $p$. Resources are rows and products are columns.  In Julia, `NamedArray` attaches labels to the numerical matrix, so `requirements[:wood, :football]` reads naturally.

A NamedArray stores input data; the JuMP variable container stores decision variables. They have distinct roles even though both support labeled indexing. Every coefficient, including zero requirements, is supplied explicitly.

There are two important arguments to NamedArray, the first in the Julia array, the second is a tuple containing the  (named indices of rows, named indices for columns).

An optional third argument gives the dimensions themselves names


In [ ]:
using NamedArrays

resources = [:wood, :plaques, :brass_football, :brass_soccer]
available = Dict(:wood => 4800, :plaques => 1750,
                 :brass_football => 1000, :brass_soccer => 1500)

# Rows: wood, plaques, brass footballs, brass soccer balls.
# Columns: football trophies, soccer trophies.
requirements = NamedArray(
    [4 2;
     1 1;
     1 0;
     0 1],
    (resources, products),
    ("resource", "product"),
)
requirements

With all inventories represented as resources, the model becomes

$$
\begin{aligned}
\max_x\quad &\sum_{p\in P}c_p x_p\\
\text{subject to}\quad &\sum_{p\in P}a_{rp}x_p\le b_r && \forall r\in R,\\
&x_p\ge0 && \forall p\in P.
\end{aligned}
$$

The brass-football row gives $x_{\text{football}}\le1000$ and the brass-soccer row gives $x_{\text{soccer}}\le1500$. These replace the upper bounds used in the preceding formulation.  (So the matrix $A$ has 4 rows).

One indexed constraint declaration now covers every resource. Changing the number of resources or products requires corresponding changes to the data, but no new resource-specific constraint lines.

In [ ]:
resource_model = Model(HiGHS.Optimizer)
set_silent(resource_model)
@variable(resource_model, x[p in products] >= 0)
@expression(resource_model, resource_use[r in resources],
    sum(requirements[r, p] * x[p] for p in products))
@constraint(resource_model, resource_limit[r in resources],
    resource_use[r] <= available[r])
@objective(resource_model, Max, sum(profit[p] * x[p] for p in products))

print(resource_model)

In [ ]:
resource_profit = solve_and_report(resource_model)
for p in products
    println(p, " trophies: ", value(x[p]))
end
println("Resource use and slack (in each resource's units):")
for r in resources
    used = value(resource_use[r])
    println(r, ": used = ", used, ", available = ", available[r],
            ", slack = ", available[r] - used)
end

total_production = sum(value(x[p]) for p in products)
for p in products
    share = 100 * value(x[p]) / total_production
    println(p, " share of production: ", round(share; digits=2), "%")
end

## 5. Optional experiment: more plaques

This experiment **changes the instance**: increase plaques from 1,750 to 1,800 and leave other data unchanged. Predict whether profit will increase before solving.

Copy the availability dictionary to preserve the baseline. Editing ordinary Julia data does not automatically update a model already constructed from it. So, we need to build a fresh model from the scenario data using the same indexed equations.

In [ ]:
scenario_available = copy(available)
scenario_available[:plaques] = 1800

scenario_model = Model(HiGHS.Optimizer)
set_silent(scenario_model)
@variable(scenario_model, x[p in products] >= 0)
@constraint(scenario_model, resource_limit[r in resources],
    sum(requirements[r, p] * x[p] for p in products) <= scenario_available[r])
@objective(scenario_model, Max,
    sum(profit[p] * x[p] for p in products))

scenario_profit = solve_and_report(scenario_model)
for p in products
    println(p, " trophies: ", value(x[p]))
end
println("Additional profit: ", scenario_profit - resource_profit)


The new optimum is 600 football trophies and 1,200 soccer trophies, earning 18,000 dollars. Additional plaques allow production to shift toward soccer trophies, which use less wood. Wood and plaques remain binding. This is a changed-data result, not a disagreement between formulations.

## What We Learned

- Start with an explicit translation of the mathematics.
- Do not code Julia/JuMP until you have an exact description of your mathematical model!
- Separate numerical data from JuMP model objects. A labeled recipe matrix connects the data table directly to indexed algebra.
- Solve, check, and only then read results. A small helper can enforce this sequence consistently.
